# TP3. Polynômes et fonctions

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp3-polynomes-fonctions/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : évaluation naïve

Écrivez `evaluer_naif(P, x)` qui calcule $P(x)$ en suivant la définition, avec une
puissance par terme.

**Résultat attendu :** `evaluer_naif([2, 3, 0, 5], 2)` vaut `48`, car
$2 + 6 + 40 = 48$.


In [ ]:
import math

def evaluer_naif(P, x):
    """Calcule P(x) en suivant la définition (une puissance par terme)."""
    total = 0
    for i in range(len(P)):
        total = total + P[i] * x ** i
    return total


print("TP3 - Ex 1.1 :", evaluer_naif([2, 3, 0, 5], 2))


## Exercice 2.1 : Horner

Écrivez `evaluer(P, x)` par le schéma de Horner, et vérifiez qu'elle donne le même
résultat que la version naïve sur plusieurs points.

**Résultat attendu :** les deux fonctions coïncident, et Horner est plus rapide sur un
polynôme de degré élevé.


In [ ]:
import math

def evaluer(P, x):
    """Calcule P(x) par le schéma de Horner."""
    total = 0
    for coefficient in reversed(P):   
        total = total * x + coefficient
    return total


print("TP3 - Ex 2.1 :")
P = [2, 3, 0, 5, -1, 4]
for x in (0, 1, 2, -3, 0.5):
    print("x =", x, ":", evaluer_naif(P, x), evaluer(P, x))


## Exercice 2.2 : mesurer, et se méfier de ses prévisions

Construisez des polynômes dont tous les coefficients valent 1, de degrés 1 000, 5 000
et 20 000, et comparez le temps des deux méthodes en `x = 1.0001`. Avant de lancer la
mesure, écrivez votre prévision : de combien Horner devrait-il être plus rapide ?

**Résultat attendu :** un rapport d'environ 2, **constant** quand le degré augmente.
Ce n'est probablement pas ce que vous aviez prévu.


In [ ]:
import math

def mesurer(fonction, P, x, repetitions=5):
    """Temps moyen d'un appel de fonction(P, x)."""
    debut = time.perf_counter()
    for r in range(repetitions):
        fonction(P, x)
    return (time.perf_counter() - debut) / repetitions


print("TP3 - Ex 2.2 : x = 1.0001")
for degre in (1000, 5000, 20000):
    P = [1] * (degre + 1)
    t_naif = mesurer(evaluer_naif, P, 1.0001)
    t_horner = mesurer(evaluer, P, 1.0001)
    print("degré", degre, ": rapport naïf/Horner =", round(t_naif / t_horner, 2))


## Exercice 2.3 : le cas où l'écart explose vraiment

Refaites la mesure en évaluant en `x = 3`, un **entier**, sur des degrés 200, 400
et 800. Le rapport reste-t-il constant ?

**Résultat attendu :** cette fois le rapport grandit avec le degré, d'environ 4 à
environ 6.


In [ ]:
import math

print("TP3 - Ex 2.3 : x = 3 (entier)")
for degre in (200, 400, 800):
    P = [1] * (degre + 1)
    t_naif = mesurer(evaluer_naif, P, 3)
    t_horner = mesurer(evaluer, P, 3)
    print("degré", degre, ": rapport naïf/Horner =", round(t_naif / t_horner, 2))


## Exercice 3.1 : la dérivée

Écrivez `deriver(P)` qui renvoie la liste des coefficients du polynôme dérivé.

**Résultat attendu :** `deriver([2, 3, 0, 5])` vaut `[3, 0, 15]`, c'est-à-dire
$3 + 15x^{2}$.


In [ ]:
import math

def deriver(P):
    """Coefficients du polynôme dérivé."""
    resultat = []
    for i in range(1, len(P)):
        resultat.append(i * P[i])
    return resultat


print("TP3 - Ex 3.1 :", deriver([2, 3, 0, 5]))

## Exercice 4.1 : dichotomie

Écrivez `dichotomie(f, a, b, tolerance)` qui renvoie une racine de `f` entre `a` et
`b`. Appliquez-la à $x^{2} - 2$ sur $[0, 2]$.

**Résultat attendu :** une valeur proche de $\sqrt{2} \approx 1{,}414214$.


In [ ]:
import math

def dichotomie(f, a, b, tolerance=1e-10):
    """Racine de f entre a et b (f(a) et f(b) de signes opposés)."""
    while b - a > tolerance:
        milieu = (a + b) / 2
        if f(a) * f(milieu) <= 0:
            b = milieu   
        else:
            a = milieu   
    return (a + b) / 2


def f(x):
    return x ** 2 - 2


print("TP3 - Ex 4.1 :", dichotomie(f, 0, 2), "(racine de 2 =", math.sqrt(2), ")")


## Exercice 4.2 : Newton sur un polynôme

Écrivez `newton(P, depart, tolerance)` qui cherche une racine du polynôme `P` en
utilisant `evaluer` et `deriver`. Appliquez-la à $x^{3} - 2x - 5$, en partant de 2.

**Résultat attendu :** environ `2.0945515`, atteint en cinq tours ou moins.


In [ ]:
import math

def newton(P, depart, tolerance=1e-10, max_tours=100):
    """Cherche une racine de P. Renvoie (racine, tours), ou (None, tours)
    si la méthode ne converge pas."""
    dP = deriver(P)
    x = depart
    for tours in range(1, max_tours + 1):
        pente = evaluer(dP, x)
        if pente == 0:
            return None, tours    
        nouveau = x - evaluer(P, x) / pente
        if abs(nouveau - x) < tolerance:
            return nouveau, tours
        x = nouveau
    return None, max_tours


racine, tours = newton([-5, -2, 0, 1], 2)  
print("TP3 - Ex 4.2 : racine =", racine, "en", tours, "tours")


## Exercice 4.3 : faire échouer Newton

Appliquez `newton` au polynôme $x^{3} - 2x + 2$, c'est-à-dire `[2, -2, 0, 1]`, en
partant de `0`. Affichez le trajet des premiers itérés pour comprendre ce qui se passe.

**Résultat attendu :** aucune convergence, et un trajet qui alterne entre `0` et `1`
indéfiniment.


In [ ]:
import math

P = [2, -2, 0, 1]  
x = 0
print("TP3 - Ex 4.3 : trajet des itérés depuis 0")
for tour in range(8):
    print("  x =", x)
    x = x - evaluer(P, x) / evaluer(deriver(P), x)
print("newton :", newton(P, 0))


## Exercice 4.4 : le prix d'un mauvais point de départ

Reprenez $x^{3} - 2x - 5$ et comparez le nombre de tours de `newton` en partant
de `2`, puis de `0`.

**Résultat attendu :** cinq tours depuis `2`, une vingtaine depuis `0`.


In [ ]:
import math

P = [-5, -2, 0, 1]
for depart in (2, 0):
    racine, tours = newton(P, depart)
    print("TP3 - Ex 4.4 : départ", depart, "->", racine, "en", tours, "tours")


## Exercice 5.1 : lire le graphique

Sur le tracé, repérez la racine trouvée par Newton. Combien de fois la courbe
coupe-t-elle l'axe des abscisses ? Ce polynôme est de degré 3 : que dire de ses autres
racines ?


In [ ]:
import math

P = [-5, -2, 0, 1]    
racine, tours = newton(P, 2)
print("TP3 - Ex 5.1 : racine de Newton =", racine)

coupures = 0
x = -5.0
while x < 5:
    if evaluer(P, x) * evaluer(P, x + 0.001) < 0:
        coupures = coupures + 1
    x = x + 0.001
print("La courbe coupe l'axe", coupures, "fois")
print("Un polynôme de degré 3 a 3 racines : 1 réelle, les 2 autres sont complexes.")

try:
    import matplotlib.pyplot as plt
    xs = [-3 + 0.05 * k for k in range(161)]
    ys = [evaluer(P, x) for x in xs]
    plt.plot(xs, ys)
    plt.axhline(0, color="black")
    plt.plot([racine], [0], "ro")
    plt.title("x^3 - 2x - 5")
    plt.show()
except ImportError:
    print("matplotlib n'est pas disponible : tracé ignoré.")


## Exercice 6.1 : somme et produit

Écrivez `somme(P, Q)` et `produit_poly(P, Q)`. Attention, les deux listes n'ont pas
forcément la même longueur.

**Résultat attendu :** `produit_poly([1, 1], [1, 1])` vaut `[1, 2, 1]`, c'est-à-dire
$(1 + x)^{2} = 1 + 2x + x^{2}$, et `produit_poly([-1, 1], [1, 1])` vaut
`[-1, 0, 1]`, l'identité $(x - 1)(x + 1) = x^{2} - 1$.


In [ ]:
import math

def somme(P, Q):
    """Somme de deux polynômes (longueurs éventuellement différentes)."""
    n = max(len(P), len(Q))
    resultat = []
    for i in range(n):
        a = P[i] if i < len(P) else 0
        b = Q[i] if i < len(Q) else 0
        resultat.append(a + b)
    return resultat


def produit_poly(P, Q):
    """Produit de deux polynômes."""
    resultat = [0] * (len(P) + len(Q) - 1)
    for i in range(len(P)):
        for j in range(len(Q)):
            resultat[i + j] = resultat[i + j] + P[i] * Q[j]
    return resultat


print("TP3 - Ex 6.1 :", somme([1, 2], [3, 4, 5]))
print(produit_poly([1, 1], [1, 1]))
print(produit_poly([-1, 1], [1, 1]))


## Exercice 6.2 : le triangle de Pascal, par les polynômes

Calculez $(1 + x)^{10}$ en multipliant dix fois `[1, 1]` par lui-même. Que
reconnaissez-vous dans la liste obtenue ?

**Résultat attendu :** `[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1]`.


In [ ]:
resultat = [1]
for k in range(10):
    resultat = produit_poly(resultat, [1, 1])
print("TP3 - Ex 6.2 :", resultat)


## Exercice 7.1 : trouver le meilleur pas

Écrivez `derivee_avant(f, x, h)` et `derivee_centree(f, x, h)`. Calculez l'erreur de
chacune sur $\sin$ en $x = 1$, dont la dérivée exacte est $\cos 1$, pour
`h` valant $10^{-1}, 10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}, 10^{-10}, 10^{-12}$.

**Résultat attendu :** l'erreur diminue puis **remonte**. Le meilleur pas est de
l'ordre de $10^{-8}$ pour la formule avant, $10^{-6}$ pour la formule centrée,
et à $h = 10^{-12}$ l'erreur est remontée de quatre à six ordres de grandeur
par rapport à ce meilleur pas.


In [ ]:
import math


def derivee_avant(f, x, h):
    return (f(x + h) - f(x)) / h


def derivee_centree(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)


exacte = math.cos(1)
print("TP3 - Ex 7.1 :")
print("       h |  erreur avant | erreur centrée")
for puissance in (1, 2, 4, 6, 8, 10, 12):
    h = 10.0 ** (-puissance)
    e_avant = abs(derivee_avant(math.sin, 1, h) - exacte)
    e_centree = abs(derivee_centree(math.sin, 1, h) - exacte)
    print(f"{h:8.0e} | {e_avant:13.2e} | {e_centree:13.2e}")


## Exercice 8.1 : les trois méthodes

Écrivez `rectangles(f, a, b, n)`, `trapezes(f, a, b, n)` et `simpson(f, a, b, n)`.
Testez-les sur $\int_0^1 x^{2}\,dx = 1/3$ pour `n` valant 10, 100 et 1000.

**Résultat attendu :** l'erreur des rectangles est divisée par 10 quand `n` est
multiplié par 10, celle des trapèzes par 100, et Simpson donne le résultat exact, à
un arrondi de $10^{-17}$ près.


In [ ]:
import math

def rectangles(f, a, b, n):
    h = (b - a) / n
    total = 0
    for i in range(n):
        total = total + f(a + i * h)
    return h * total


def trapezes(f, a, b, n):
    h = (b - a) / n
    total = (f(a) + f(b)) / 2
    for i in range(1, n):
        total = total + f(a + i * h)
    return h * total


def simpson(f, a, b, n):
    """n doit être pair."""
    h = (b - a) / n
    total = f(a) + f(b)
    for i in range(1, n):
        if i % 2 == 1:
            total = total + 4 * f(a + i * h)
        else:
            total = total + 2 * f(a + i * h)
    return h * total / 3


def carre(x):
    return x ** 2


print("TP3 - Ex 8.1 : intégrale de x^2 sur [0, 1] (exact : 1/3)")
for n in (10, 100, 1000):
    e1 = abs(rectangles(carre, 0, 1, n) - 1 / 3)
    e2 = abs(trapezes(carre, 0, 1, n) - 1 / 3)
    e3 = abs(simpson(carre, 0, 1, n) - 1 / 3)
    print(f"n = {n:4d} | rectangles {e1:.2e} | trapèzes {e2:.2e} | Simpson {e3:.2e}")


## Exercice 8.2 : calculer π

On a $\int_0^1 \frac{4}{1 + x^{2}}\,dx = \pi$. Calculez cette intégrale par les
trapèzes et par Simpson avec `n = 1000`, et comparez à `math.pi`.

**Résultat attendu :** une erreur de l'ordre de $10^{-7}$ pour les trapèzes, et
un résultat indistinguable de `math.pi` pour Simpson.


In [ ]:
import math

def g(x):
    return 4 / (1 + x ** 2)


print("TP3 - Ex 8.2 :")
print("trapèzes :", trapezes(g, 0, 1, 1000), "erreur", abs(trapezes(g, 0, 1, 1000) - math.pi))
print("Simpson  :", simpson(g, 0, 1, 1000), "erreur", abs(simpson(g, 0, 1, 1000) - math.pi))
print("math.pi  :", math.pi)